In [ ]:
!pip install librosa

In [ ]:
import os

# base_path = "/path to your video folder's parent folder"

base_path = os.getcwd()


current_folder = "/clarks_session"


video1_input = "/input1.mp4" # speaker video
video2_input = "/input2.mp4" # presentation

In [ ]:
# VIDEO TO AUDIO
import subprocess

subprocess.run([
    "ffmpeg",
    "-i", base_path+current_folder+video1_input,
    "-vn",
    "-c:a", "mp3",
    "-b:a", "320k",
    base_path+current_folder+"/output1.mp3"
])

subprocess.run([
    "ffmpeg",
    "-i", base_path+current_folder+video2_input,
    "-vn",
    "-c:a", "mp3",
    "-b:a", "320k",
    base_path+current_folder+"/output2.mp3"
])

In [ ]:
import subprocess

subprocess.run([
    "ffmpeg",
    "-i", base_path+current_folder+"/output1.mp3",
    "-ac", "1",
    "-ar", "8000",
    base_path+current_folder+"/audio1.wav"
])

subprocess.run([
    "ffmpeg",
    "-i", base_path+current_folder+"/output2.mp3",
    "-ac", "1",
    "-ar", "8000",
    base_path+current_folder+"/audio2.wav"
])

In [ ]:
import librosa

import numpy as np

y1, sr = librosa.load(base_path+current_folder+"/audio1.wav", sr=8000, mono=True)

y2, sr = librosa.load(base_path+current_folder+"/audio2.wav", sr=8000, mono=True)

def envelope(y, frame=1024, hop=512):

    return librosa.feature.rms(y=y, frame_length=frame, hop_length=hop)[0]

e1 = envelope(y1)

e2 = envelope(y2)

corr = np.correlate(e1, e2, mode="full")

lag = np.argmax(corr) - len(e2)

offset_seconds = lag * (512 / sr)

print(offset_seconds)

In [ ]:
import librosa
import numpy as np
import subprocess
import os

abs_offset_str = f"{abs(offset_seconds):.6f}"

if offset_seconds < 0:
    # Video 2 is early, then Trim the beginning of Video 2, keep Video 1 as-is
    print(f"Negative offset. Trimming {abs_offset_str}s from {video2_input}...")
    
    # Trim Video 2
    cmd_trim_2 = [
        'ffmpeg', '-y',
        '-ss', abs_offset_str,    # Seek forward by the offset amount (Trims the start)
        '-i', base_path+current_folder+video2_input,
        '-c', 'copy',             # Stream copy (Fast, no quality loss)
        base_path+current_folder+'/video2_synced.mp4'
    ]
    subprocess.run(cmd_trim_2, check=True)

    os.rename(base_path+current_folder+video1_input, base_path+current_folder+"/video1_synced.mp4")

else:
    # Video 1 is early, then Trim the beginning of Video 1, keep Video 2 as-is
    print(f"Positive offset. Trimming {abs_offset_str}s from {video1_input}...")
    
    # Trim Video 1
    cmd_trim_1 = [
        'ffmpeg', '-y',
        '-ss', abs_offset_str,    # Seek forward by the offset amount (Trims the start)
        '-i', base_path+current_folder+video1_input,
        '-c', 'copy',             # Stream copy (Fast, no quality loss)
        base_path+current_folder+'/video1_synced.mp4'
    ]
    subprocess.run(cmd_trim_1, check=True)

    os.rename(base_path+current_folder+video2_input, base_path+current_folder+"/video2_synced.mp4")

print("Synchronization complete! Check video1_synced.mp4 and video2_synced.mp4.")